# 08 · 실시간 통합 (DA + FastSAM + ArUco) · 자연 웹캠 시점 합성

각 도구를 강점에만 사용해 실시간으로:

| 역할 | 도구 |
|---|---|
| 물체 감지 + 선/누움 판별 | **Depth Anything V2 높이맵** |
| 완전한 실루엣(경계) | **FastSAM** |
| 정확한 높이·길이·위치 | **ArUco 기하** |

**표시:** 정면(top-down) 변환 없이 **원본 웹캠 화면 그대로**, 물체 부위에 **높이 히트맵을 반투명 합성** + 윤곽/측정 라벨 + 우상단 높이맵 인셋.

**측정:** 선 물체 = 수직모서리 높이 H(±10% 수준), 누운 물체 = 전체 실루엣 길이 L. (지름/부피 정밀은 다중시점 필요.)

> 성능: DA와 FastSAM을 **매 프레임 2개** 돌려서 GTX1660에서 수 fps. 느리면 `imgsz`↓.

In [1]:
import os, sys, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, depth_volume as dv, live_combined as lc
from ultralytics import FastSAM
print('OpenCV', cv2.__version__)

OUTPUT_DIR = os.path.join(ROOT, 'output')
SCENE_DIR = os.path.join(ROOT, 'data', 'scene_images')
SQ = 0.038
board = cv2.aruco.CharucoBoard((5, 7), SQ, SQ*22/30,
                               cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000))
K, dist = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))

CAM_INDEX = 0
pipe = dv.load_depth_model('depth-anything/Depth-Anything-V2-Small-hf', device=0)
model = FastSAM('FastSAM-s.pt')
print('models ready')

OpenCV 4.13.0


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

models ready


## 정지영상 합성 미리보기 (웹캠 없이)

In [ ]:
cands = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'snap_raw_*.png'))) or \
        sorted(glob.glob(os.path.join(SCENE_DIR, '*.*')))
img = cv2.imread(cands[-1]); print('대상:', os.path.basename(cands[-1]))
vis, objs, markers = lc.process_frame_combined(img, pipe, model, board, K, dist, square_len=SQ, imgsz=1024)
for i, o in enumerate(objs):
    print(f"  #{i} {o['type']:6s} {o['label']}  center={tuple(round(v) for v in o['center_mm'])}mm")
plt.figure(figsize=(14, 8)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## 실시간 실행

보드 위에 물체를 올리고 실행. 별도 창에 **자연 시점 합성 화면**이 뜬다.
- **`s`**: 스냅(raw+vis) 저장  ·  **`q`**: 종료

> 배경차분 방식과 달리 **빈 보드 기준(`r`)이 필요 없다** — 물체를 '평면 위로 솟은 것'으로 깊이가 직접 감지하니까.

In [3]:
lc.run_live_combined(K, dist, board, square_len=SQ, pipe=pipe, model=model,
                     cam_index=CAM_INDEX, imgsz=640, snapshot_dir=OUTPUT_DIR)
print('종료됨')

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


snapshot 0
종료됨


## 튜닝 & 다음

- **느리면**: `imgsz`(FastSAM)↓ (512), 웹캠 해상도↓. DA는 고정 비용.
- **물체 놓침/잔검출**: `min_height_mm`(솟음 임계)·`min_area_px` 조정.
- **지름/부피 정확도**: 단일 시점 한계 → 다중시점(다른 각도 2장 ArUco 정합)으로 확장.
- **높이 정확**(±10%): 선 물체는 수직모서리 기하로 잘 나옴. 누운 물체 길이는 끝단 가림만큼 −.